[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jooyongsim/Pendulum_control/blob/sumin/BC_Pendulum/bc_rotary_swingup_colab.ipynb)

# Behavior Cloning + DAgger — 회전형 진자 스윙업 (시뮬레이션)

신경망이 **우리가 만든 스윙업 제어기(에너지 펌핑 + 극 배치)** 를 흉내 내도록 학습합니다.
시뮬레이터는 실제 장비에서 측정한 모델입니다 (ω_n 7.499, ζ 0.0117, b 0.730, 모터 한계 47.1 rad/s², 50 Hz 정책, 4 ms 지연, 0.3° 엔코더).

| 단계 | 내용 | 강의 |
|---|---|---|
| 1 | 선생님(전문가) 제어기 확인 | |
| 2 | **시연 데이터 수집** — (관측, 선생님 행동) 쌍 | Lec06 imitation learning |
| 3 | **BC 학습** — MSE 회귀 | Lec05: 가우시안 최대우도 = MSE |
| 4 | compounding error 확인 | Lec06 covariate shift |
| 5 | **DART** — 노이즈 섞어 시연 | 교정 데이터 |
| 6 | **DAgger** — 학생이 간 상태에 선생님 정답 추가 | 교정 데이터 |
| 7 | 비교 · 애니메이션 | |
| 8 | weight 저장 → 실제 장비 | |

위에서부터 차례로 실행하세요. GPU는 필요 없습니다.

## 0. 코드 준비
시뮬레이터(`rotary_model.py`)와 BC 도구(`bc_rotary_swingup.py`)를 파일로 씁니다. **실행만 하면 됩니다.**

In [ ]:
%%writefile rotary_model.py
"""Rotary inverted pendulum (STEVAL-EDUKIT01) under acceleration control: plant, motor, controllers.

Shared by rotary_balance_realtime.py and rotary_swingup_realtime.py. The numbers and control
laws are the ones running on the real rig (swing-up/balance_control.py, swing-up/swingup_control.py,
swing-up/SwingupController/SwingupController.ino).

theta = pendulum angle from UPRIGHT (0 = balanced, +-pi = hanging), phi = arm angle,
u = phi'' [rad/s^2] (the host commands the arm acceleration).

    theta'' = (3g/2l) sin(theta) + b cos(theta) phi'' + sin(theta) cos(theta) phi'^2
              - 2 zeta wn theta' - Fc sign(theta')
    b = 3r/2l

Identified on our unit: wn = 7.499 rad/s (3g/2l = wn^2, l = 262 mm), zeta = 0.01172,
dry friction 2.705 deg/s of amplitude, b = 0.730 (r = 127 mm) from the open-loop test.

Motor = the firmware, not an ideal acceleration source:
    v += a dt           |a| <= 24000 pps^2, |v| <= 4000 pps   (target velocity)
    |v| < 30 pps        motor stopped (L6474 minimum run speed)
    |phi| >= 150 deg    latched stop
Measurement: 100 Hz loop, 4 ms from reading to the new command, 0.3 deg encoder,
arm angle in whole microsteps, theta' by filtered differencing.
"""
import math
from collections import deque

import numpy as np

# ---------------------------------------------------------------- plant (identified)
G = 9.81
WN, ZETA = 7.499, 0.01172
L = 3 * G / (2 * WN ** 2)                       # 0.262 m, uniform rod
A_G = WN ** 2                                   # 3g/2l [1/s^2]
C_D = 2 * ZETA * WN                             # viscous [1/s]
B_U = 0.730                                     # 3r/2l from the open-loop test
R_ARM = 2 * L * B_U / 3                         # 0.127 m
COULOMB_DPS = 2.705                             # amplitude lost per second to dry friction
F_C = math.radians(COULOMB_DPS) * WN ** 2 * (2 * math.pi / WN) / 4.0   # [rad/s^2]

# ---------------------------------------------------------------- motor (SwingupController.ino)
STEP_RAD = 2 * math.pi / 3200                   # 1/16 microstep
ACCEL_MAX = 24000 * STEP_RAD                    # 47.1 rad/s^2
SPEED_MAX = 4000 * STEP_RAD                     # 7.85 rad/s
SPEED_MIN = 30 * STEP_RAD                       # below this the motor does not run
ARM_LATCH = math.radians(150.0)

# ---------------------------------------------------------------- loop / sensor
CONTROL_HZ = 100.0
DELAY = 0.004                                   # s, reading -> new command
ENC_RAD = math.radians(0.3)                     # 1200 CPR
TAU_V = 0.01                                    # s, theta' filter
DT = 0.001                                      # plant step

# ---------------------------------------------------------------- controllers (hardware defaults)
POLES = [-6 + 6j, -6 - 6j, -1.5 + 1j, -1.5 - 1j]
Q_LQR, R_LQR = [150.0, 5.0, 5.0, 1.0], 0.5      # Lec03 slide 23, in our state order
SWING = dict(pump=30.0, ke=0.3, etarget=1.0, kpa=3.0, kda=5.0,
             catch=12.0, catch_w=4.0, catch_arm=60.0, fall=30.0, bias_tau=6.0)


def wrap(a):
    return (a + math.pi) % (2 * math.pi) - math.pi


def state_matrices(b=B_U):
    A = np.array([[0, 1, 0, 0], [A_G, -C_D, 0, 0], [0, 0, 0, 1], [0, 0, 0, 0]], float)
    B = np.array([[0], [b], [0], [1]], float)
    return A, B


def place4(poles=POLES, b=B_U):
    """Ackermann: K = e_n^T C^-1 phi(A), u = -K [theta, theta', phi, phi']."""
    A, B = state_matrices(b)
    ctrb = np.hstack([np.linalg.matrix_power(A, i) @ B for i in range(4)])
    phi = np.zeros_like(A)
    for c in np.real(np.poly(poles)):
        phi = phi @ A + c * np.eye(4)
    return np.linalg.solve(ctrb.T, np.eye(4)[:, -1]) @ phi


def place2(wn=12.0, zeta=0.7, b=B_U):
    """Rod only: s^2 + (c + b k2) s + (b k1 - a) = s^2 + 2 zeta wn s + wn^2. The arm is ignored."""
    return np.array([(wn ** 2 + A_G) / b, (2 * zeta * wn - C_D) / b, 0.0, 0.0])


def lqr4(q=Q_LQR, r=R_LQR, b=B_U):
    """LQR gain (Lec03): minimise J = int(x' diag(q) x + r u^2) dt. The Riccati solution comes
    from the Hamiltonian's stable eigenvectors, so numpy is enough."""
    A, B = state_matrices(b)
    H = np.block([[A, -B @ B.T / r], [-np.diag(np.asarray(q, float)), -A.T]])
    w, v = np.linalg.eig(H)
    stable = v[:, np.argsort(w.real)[:4]]
    P = np.real(stable[4:] @ np.linalg.inv(stable[:4]))
    return (B.T @ P / r).ravel()


def energy(th, w):
    """0 when upright at rest, -2 A_G (= -112) when hanging at rest."""
    return 0.5 * w * w + A_G * (math.cos(th) - 1)


class Plant:
    """x = [theta, theta', phi, v]; v is the firmware's target velocity, integrated from the command."""

    def __init__(self, theta0=0.0, b=B_U, coulomb=True, motor_limits=True):
        self.b, self.coulomb, self.motor_limits = b, coulomb, motor_limits
        self.reset(theta0)

    def reset(self, theta0=0.0):
        self.x = np.array([theta0, 0.0, 0.0, 0.0])
        self.a = 0.0                                # applied arm acceleration command
        self.t = 0.0
        self.latched = False

    def running(self, v):
        return (not self.motor_limits) or abs(v) >= SPEED_MIN

    def deriv(self, x, a):
        th, w, phi, v = x
        on = self.running(v)
        acc = a if on else 0.0
        phid = v if on else 0.0
        dw = (A_G * math.sin(th) + self.b * math.cos(th) * acc
              + math.sin(th) * math.cos(th) * phid * phid - C_D * w)
        if self.coulomb:
            dw -= F_C * math.tanh(w / 0.01)
        return np.array([w, dw, phid, a])

    def step(self):
        a = 0.0 if self.latched else self.a
        if self.motor_limits:
            a = max(-ACCEL_MAX, min(ACCEL_MAX, a))
        x = self.x
        k1 = self.deriv(x, a)
        k2 = self.deriv(x + DT / 2 * k1, a)
        k3 = self.deriv(x + DT / 2 * k2, a)
        k4 = self.deriv(x + DT * k3, a)
        self.x = x + DT / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        self.t += DT
        if self.motor_limits:
            self.x[3] = max(-SPEED_MAX, min(SPEED_MAX, self.x[3]))
            if abs(self.x[2]) >= ARM_LATCH and not self.latched:
                self.latched = True
                self.x[3] = 0.0

    @property
    def arm_rate(self):
        v = self.x[3]
        return v if self.running(v) else 0.0

    def kick(self, dw):
        self.x[1] += dw


class Sensor:
    """What the host sees: quantised encoder, whole microsteps, filtered theta'."""

    def __init__(self, realistic=True):
        self.realistic = realistic
        self.prev = None
        self.w = 0.0

    def reset(self):
        self.prev, self.w = None, 0.0

    def read(self, plant, offset=0.0):
        """offset = encoder zero error: the true vertical reads as `offset`."""
        th, w, phi, _ = plant.x
        if not self.realistic:
            return np.array([wrap(th + offset), w, phi, plant.arm_rate])
        m = round((th + offset) / ENC_RAD) * ENC_RAD
        ts = 1.0 / CONTROL_HZ
        if self.prev is not None:
            self.w += (ts / (TAU_V + ts)) * (wrap(m - self.prev) / ts - self.w)
        self.prev = m
        return np.array([wrap(m), self.w, round(phi / STEP_RAD) * STEP_RAD, plant.arm_rate])


class Balancer:
    """u = -K x. Gives up beyond `guard`."""

    def __init__(self, K, guard_deg=25.0):
        self.K, self.guard = np.asarray(K, float), math.radians(guard_deg)
        self.status = "OK"

    def reset(self):
        self.status = "OK"

    def __call__(self, z, t):
        if abs(z[0]) > self.guard:
            self.status = "FALLEN"
        return 0.0 if self.status != "OK" else float(-self.K @ z)


class SwingUp:
    """Same state machine as swing-up/swingup_control.py.

    swing   : +pump 0.15 s, -pump 0.15 s, then
              u = pump * clip(kE (E_target - E), 0, 1) * sign(theta' cos theta) - kda phi' - kpa phi
    catch   : |theta| < catch, |theta'| < catch_w, |phi| < catch_arm -> 4-state balancer,
              arm reference = arm angle at the catch, decaying with 3 s
    balance : slow upright-offset adaptation; beyond `fall` back to swing
    """

    def __init__(self, K, **kw):
        self.K = np.asarray(K, float)
        self.p = dict(SWING, **kw)
        self.gamma = abs(self.K[2]) / (self.K[0] * self.p["bias_tau"]) if self.p["bias_tau"] > 0 else 0.0
        self.reset()

    def reset(self, offset_guess=0.0):
        self.mode, self.phi_ref, self.b_hat = "swing", 0.0, offset_guess
        self.t_start = None
        self.t_catch, self.catches, self.falls = None, 0, 0
        self.E = float("nan")

    def __call__(self, z, t):
        p, ts = self.p, 1.0 / CONTROL_HZ
        if self.t_start is None:
            self.t_start = t
        tr = t - self.t_start
        th_raw, w, phi, phid = z
        th = wrap(th_raw - self.b_hat)
        self.E = energy(th, w)
        if (self.mode == "swing" and tr > 0.3 and abs(math.degrees(th)) < p["catch"]
                and abs(w) < p["catch_w"] and abs(math.degrees(phi)) < p["catch_arm"]):
            self.mode, self.phi_ref = "balance", phi
            self.catches += 1
            if self.t_catch is None:
                self.t_catch = tr
        elif self.mode == "balance" and abs(math.degrees(th)) > p["fall"]:
            self.mode = "swing"
            self.falls += 1
        if self.mode == "swing":
            if tr < 0.15:
                return p["pump"]
            if tr < 0.30:
                return -p["pump"]
            scale = min(1.0, max(0.0, p["ke"] * (p["etarget"] - self.E)))
            sgn = 1.0 if w * math.cos(th) > 0 else -1.0
            return p["pump"] * scale * sgn - p["kda"] * phid - p["kpa"] * phi
        self.phi_ref *= math.exp(-ts / 3.0)
        self.b_hat += self.gamma * (phi - self.phi_ref) * ts
        self.b_hat = max(-math.radians(5), min(math.radians(5), self.b_hat))
        return float(-self.K @ np.array([th, w, phi - self.phi_ref, phid]))


class Loop:
    """Plant at 1 kHz, controller at 100 Hz, each command applied DELAY after its reading."""

    def __init__(self, plant, controller, sensor=None, arm_limit_deg=140.0, offset=0.0):
        self.plant, self.ctrl = plant, controller
        self.sensor = sensor or Sensor()
        self.arm_limit = math.radians(arm_limit_deg)
        self.offset = offset                        # encoder zero error: true vertical reads as this
        self.on = True
        self.reset()

    def reset(self):
        self.next_ctrl = 0.0
        self.queue = deque()                        # (apply time, command)
        self.u = 0.0
        self.stop = ""
        self.sensor.reset()

    def _check_stop(self):
        pl = self.plant
        if pl.latched:
            self.stop = "FIRMWARE LATCH (150 deg)"
        elif abs(pl.x[2]) > self.arm_limit:
            self.stop = f"ARM LIMIT ({math.degrees(self.arm_limit):.0f} deg)"
        if self.stop:                               # host sends ZERO_VELOCITY + HARD_STOP
            self.queue.clear()
            pl.a, pl.x[3] = 0.0, 0.0

    def advance(self, n):
        pl = self.plant
        for _ in range(n):
            if pl.t >= self.next_ctrl - 1e-9:
                self.next_ctrl += 1.0 / CONTROL_HZ
                if not self.stop:
                    self._check_stop()
                z = self.sensor.read(pl, self.offset)
                u = self.ctrl(z, pl.t) if (self.on and not self.stop) else 0.0
                self.u = max(-ACCEL_MAX, min(ACCEL_MAX, u))
                if not self.stop:
                    lag = DELAY if self.sensor.realistic else 0.0
                    self.queue.append((pl.t + lag, self.u))
            while self.queue and pl.t >= self.queue[0][0] - 1e-9:
                pl.a = self.queue.popleft()[1]
            pl.step()


In [ ]:
%%writefile bc_rotary_swingup.py
"""Behavior cloning + DAgger (Lec05/Lec06) for the rotary pendulum SWING-UP, on the identified rig model.

A neural network learns to swing the rod up, catch it and balance it -- and to swing up again after a
fall -- by imitating our hand-designed controller (energy pumping + 4-state pole placement), the
"expert". The simulation is rotary_model.py: identified plant (wn 7.499, zeta 0.0117, b 0.730,
Coulomb friction), firmware limits (47.1 rad/s^2, 7.85 rad/s, 30 pps floor, 150 deg latch),
100 Hz loop with 4 ms delay and a 0.3 deg encoder.

Policy format = swing-up/swingup_rl_control.py (--kind bc), so the trained weights run on the rig:
    obs = [cos th, sin th, th', phi, phi']      th from upright (0 = balanced), as measured
    a   = tanh(net(obs)[0]) in [-1, 1]  ->  u = a * 47.1 rad/s^2
    net = nn.Sequential(Linear 5-128, ReLU, Linear 128-128, ReLU, Linear 128-out), keys 'net.N.*'
    recomputed every 20 ms (50 Hz) and held for two 100 Hz ticks.
Inputs are normalised while training; the normalisation is folded into the first layer when saving,
so the saved network takes raw observations.

Expert (stateless, so it can label ANY state -- DAgger needs that):
    near the top (|th| < 20 deg, |th'| < 5 rad/s):  u = -K x,  K = [236.23, 26.54, -4.16, -4.55]
    otherwise:  u = 30 clip(0.3 (1 - E), 0, 1) sign(th' cos th) - 5 phi' - 3 phi,   E = th'^2/2 + wn^2 (cos th - 1)

Lecture mapping
    version 0   deterministic policy, l2 regression to the expert action  (= Gaussian MLE, Lec05)
    version 1   Gaussian policy, maximise log N(a* | mu, sigma) with learned sigma, deploy mu
    compounding errors / covariate shift -> corrective data:
    DART        expert acts with injected noise, every state labelled with the clean expert action
    DAgger      student acts (mixed with the expert at rate beta), visited states labelled by the expert

    python3 bc_rotary_swingup.py train            # collect, BC, DART, DAgger, compare, save
    python3 bc_rotary_swingup.py eval             # re-evaluate the saved models
    python3 bc_rotary_swingup.py demo             # real-time window (student vs expert)
    python3 bc_rotary_swingup.py demo --model bc
    python3 bc_rotary_swingup.py gif              # record a demo to bc_models/swingup_demo.gif

Saved weights: simulator/bc_models/rotary_bc_<name>.pt (torch, for the rig) and .npz (numpy, this file).
On the rig:  python3 swing-up/swingup_rl_control.py --model BC_Pendulum/bc_models/rotary_bc_dagger.pt --kind bc

Demo keys (click the window first):
    up : knock it over (+5 rad/s)     left / right : kick (-/+ 1 rad/s)
    m  : next controller (dagger -> bc_dart -> bc -> bc_nll -> expert)
    c  : controller on / off          r : reset (hanging)          space : pause
If the window does not open under Wayland:  MPLBACKEND=TkAgg python3 ...
"""
import argparse
import math
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import rotary_model as rm  # noqa: E402

OUT = Path(__file__).resolve().parent / "bc_models"
K = rm.place4()
SW = rm.SWING
CATCH_TH, CATCH_W = math.radians(20.0), 5.0     # no arm gate: leaving the top because the arm
                                                  # passed 60 deg made the expert drop the rod
POLICY_EVERY = 2                        # 100 Hz loop ticks per policy step -> 50 Hz
ARM_LIMIT_DEG = 140.0
HIDDEN = 128
MODELS = ("dagger", "bc_dart", "bc", "bc_nll")


# ---------------------------------------------------------------- observation and expert
def obs(z):
    th, w, phi, phid = z
    return np.array([math.cos(th), math.sin(th), w, phi, phid])


def expert_u(z):
    """Stateless version of swing-up/swingup_control.py: same gains, no mode memory."""
    th, w, phi, phid = z
    if abs(th) < CATCH_TH and abs(w) < CATCH_W:
        return float(-K @ np.asarray(z, float))
    E = 0.5 * w * w + rm.A_G * (math.cos(th) - 1.0)
    scale = min(1.0, max(0.0, SW["ke"] * (SW["etarget"] - E)))
    return SW["pump"] * scale * (1.0 if w * math.cos(th) > 0 else -1.0) - SW["kda"] * phid - SW["kpa"] * phi


def expert_action(z):
    return float(np.clip(expert_u(z) / rm.ACCEL_MAX, -1.0, 1.0))


# ---------------------------------------------------------------- policy
class Policy:
    """MLP with ReLU hidden layers, numpy forward (no torch needed to run it)."""

    def __init__(self, layers, gaussian=False):
        self.layers, self.gaussian = layers, gaussian      # [(W, b), ...] taking RAW obs

    def __call__(self, o):
        h = np.asarray(o, float)
        for i, (W, b) in enumerate(self.layers):
            h = W @ h + b
            if i < len(self.layers) - 1:
                h = np.maximum(h, 0.0)
        return float(np.tanh(h[0]))

    def save(self, stem):
        """stem.npz (numpy) and stem.pt (torch state_dict, keys net.0/2/4) -- same weights."""
        stem = Path(stem)
        stem.parent.mkdir(parents=True, exist_ok=True)
        flat = {}
        for i, (W, b) in enumerate(self.layers):
            flat[f"W{i}"], flat[f"b{i}"] = W, b
        np.savez(stem.with_suffix(".npz"), gaussian=self.gaussian, **flat)
        try:
            import collections
            import torch
            sd = collections.OrderedDict()
            for i, (W, b) in enumerate(self.layers):
                sd[f"net.{2 * i}.weight"] = torch.tensor(W, dtype=torch.float32)
                sd[f"net.{2 * i}.bias"] = torch.tensor(b, dtype=torch.float32)
            torch.save(sd, stem.with_suffix(".pt"))
        except ImportError:
            pass

    @classmethod
    def load(cls, path):
        d = np.load(Path(path).with_suffix(".npz"))
        n = len([k for k in d.files if k.startswith("W")])
        return cls([(d[f"W{i}"], d[f"b{i}"]) for i in range(n)], bool(d["gaussian"]))


def train_policy(X, Y, gaussian=False, epochs=80, lr=1e-3, seed=0, verbose=True):
    """version 0: MSE on tanh(out0).  version 1: Gaussian NLL, sigma = exp(out1)."""
    import torch
    import torch.nn as nn

    torch.manual_seed(seed)
    X, Y = np.asarray(X, np.float32), np.asarray(Y, np.float32)
    mean, std = X.mean(0), X.std(0) + 1e-6
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X))
    n_val = max(1, len(X) // 10)
    Xt = torch.tensor((X[idx] - mean) / std)
    Yt = torch.tensor(Y[idx])[:, None]
    Xtr, Ytr, Xva, Yva = Xt[n_val:], Yt[n_val:], Xt[:n_val], Yt[:n_val]
    net = nn.Sequential(nn.Linear(5, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                        nn.Linear(HIDDEN, 2 if gaussian else 1))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

    def loss_fn(xb, yb):
        out = net(xb)
        mu = torch.tanh(out[:, :1])
        if not gaussian:
            return ((mu - yb) ** 2).mean()                              # -log N(a*|mu, s) with s fixed
        log_s = out[:, 1:].clamp(-6.0, 1.0)
        return (0.5 * ((yb - mu) / log_s.exp()) ** 2 + log_s).mean()     # -log N(a*|mu, s) + const

    best, best_state = float("inf"), None
    for _ in range(epochs):
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 256):
            b = perm[i:i + 256]
            opt.zero_grad()
            loss_fn(Xtr[b], Ytr[b]).backward()
            opt.step()
        sched.step()
        with torch.no_grad():
            val = float(((torch.tanh(net(Xva)[:, :1]) - Yva) ** 2).mean())
        if val < best:
            best, best_state = val, {k: v.clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    if verbose:
        print(f"      {len(Xtr)} training samples, best validation MSE {best:.2e}")
    sd = {k: v.numpy().astype(np.float64) for k, v in net.state_dict().items()}
    W0, b0 = sd["0.weight"], sd["0.bias"]
    layers = [(W0 / std, b0 - W0 @ (mean / std)),                       # fold the normalisation in
              (sd["2.weight"], sd["2.bias"]), (sd["4.weight"], sd["4.bias"])]
    return Policy(layers, gaussian)


# ---------------------------------------------------------------- simulation
class Controller:
    """Policy step every 20 ms. Acts with `actor` (None = expert), optional noise / expert mixing,
    records (obs, expert action) at every policy step."""

    def __init__(self, actor=None, record=False, noise=0.0, beta=0.0, rng=None):
        self.actor, self.record, self.noise, self.beta = actor, record, noise, beta
        self.rng = rng or np.random.default_rng()
        self.X, self.Y = [], []
        self.k, self.u, self.a_expert = 0, 0.0, 0.0

    def __call__(self, z, t):
        if self.k % POLICY_EVERY == 0:
            o, a_exp = obs(z), expert_action(z)
            self.a_expert = a_exp
            if self.record:
                self.X.append(o); self.Y.append(a_exp)
            a = a_exp if (self.actor is None or self.rng.random() < self.beta) else self.actor(o)
            if self.noise:
                a += self.noise * self.rng.standard_normal()
            self.u = float(np.clip(a, -1.0, 1.0)) * rm.ACCEL_MAX
        self.k += 1
        return self.u


def make_loop(ctrl, th0, w0=0.0, b=rm.B_U):
    plant = rm.Plant(th0, b=b)
    plant.x[1] = w0
    return rm.Loop(plant, ctrl, rm.Sensor(True), arm_limit_deg=ARM_LIMIT_DEG)


def start_state(kind, rng):
    """'hang': hanging, small tilt -> swing-up.  'top': near upright, knocked over at 2 s -> recovery."""
    if kind == "hang":
        return math.pi + math.radians(rng.uniform(-3, 3)), rng.uniform(-0.2, 0.2), None
    return math.radians(rng.uniform(-3, 3)), 0.0, (2.0, rng.choice([-1, 1]) * rng.uniform(4.0, 6.0))


def run_episode(ctrl, kind, rng, duration=10.0, b=rm.B_U, delay=None, log=False):
    th0, w0, kick = start_state(kind, rng)
    d0 = rm.DELAY
    if delay is not None:
        rm.DELAY = delay
    try:
        lp = make_loop(ctrl, th0, w0, b)
        up_at, arm_max, kicked, rows = None, 0.0, False, []
        while lp.plant.t < duration:
            if kick and not kicked and lp.plant.t >= kick[0]:
                lp.plant.kick(kick[1]); kicked = True
            lp.advance(10)
            th = rm.wrap(lp.plant.x[0])
            arm_max = max(arm_max, abs(lp.plant.x[2]))
            if up_at is None and abs(th) < math.radians(3) and (kind == "hang" or kicked):
                up_at = lp.plant.t
            if log:
                rows.append((lp.plant.t, math.degrees(th), math.degrees(lp.plant.x[2]), lp.u,
                             ctrl.a_expert * rm.ACCEL_MAX))
    finally:
        rm.DELAY = d0
    th_end = abs(rm.wrap(lp.plant.x[0]))
    ok = not lp.stop and th_end < math.radians(3)
    out = dict(ok=ok, up=up_at, arm=math.degrees(arm_max), why=lp.stop)
    if log:
        out["log"] = np.array(rows)
    return out


def collect(n_eps, rng, actor=None, noise=0.0, beta=0.0, kinds=("hang", "top")):
    """Run episodes and return (obs, expert action) pairs from every policy step."""
    X, Y = [], []
    for i in range(n_eps):
        c = Controller(actor, record=True, noise=noise, beta=beta, rng=rng)
        run_episode(c, kinds[i % len(kinds)], rng)
        X += c.X; Y += c.Y
    return np.array(X), np.array(Y)


TESTS = {"swing-up": dict(kind="hang"),
         "knock-over": dict(kind="top"),
         "b -20%": dict(kind="hang", b=0.8 * rm.B_U),
         "delay 12 ms": dict(kind="hang", delay=0.012)}


def evaluate(actor, test, n=20, seed=123):
    rng = np.random.default_rng(seed)
    res = [run_episode(Controller(actor, rng=rng), rng=rng, **TESTS[test]) for _ in range(n)]
    ok = [r for r in res if r["ok"]]
    ups = [r["up"] for r in ok if r["up"]]
    return dict(rate=len(ok) / n, up=float(np.mean(ups)) if ups else float("nan"),
                arm=float(np.mean([r["arm"] for r in ok])) if ok else float("nan"))


def compare(models, n=20):
    names = list(TESTS)
    print(f"\n{'controller':<10}" + "".join(f"{t:>14}" for t in names) + f"{'swing-up time':>15}{'arm max':>9}")
    for name, m in [("expert", None)] + list(models.items()):
        cells, first = "", None
        for t in names:
            r = evaluate(m, t, n)
            first = first or r
            cells += f"{r['rate'] * 100:>13.0f}%"
        print(f"{name:<10}{cells}{first['up']:>14.2f}s{first['arm']:>8.0f}d")
    print(f"success = balanced (|theta| < 3 deg) at 10 s without hitting the arm limit, {n} runs each.\n"
          "knock-over: starts upright, kicked 4-6 rad/s at 2 s.  b -20% / delay 12 ms: model errors the rig could have.")


# ---------------------------------------------------------------- the full pipeline
def train_all(n_demo=20, dagger_rounds=5, dagger_eps=6, seed=0):
    rng = np.random.default_rng(seed)
    models = {}
    print(f"expert: energy pumping + pole placement (stateless), policy at {rm.CONTROL_HZ / POLICY_EVERY:.0f} Hz")

    print(f"\n[1] demonstrations: {n_demo} expert episodes (swing-up from hanging, recovery after a knock-over)")
    X0, Y0 = collect(n_demo, rng)
    print(f"      {len(X0)} (obs, action) pairs")
    print("[2] BC version 0: MSE regression")
    models["bc"] = train_policy(X0, Y0, seed=seed)

    print("[3] BC + DART: expert demos with action noise (sigma 0.2), clean labels")
    Xd, Yd = collect(n_demo, rng, noise=0.2)
    models["bc_dart"] = train_policy(np.vstack([X0, Xd]), np.concatenate([Y0, Yd]), seed=seed)

    print(f"[4] DAgger: from BC, {dagger_rounds} rounds x {dagger_eps} episodes, student acts, expert labels")
    X, Y, pol = X0.copy(), Y0.copy(), models["bc"]
    for i in range(dagger_rounds):
        beta = 0.5 ** (i + 1)
        Xn, Yn = collect(dagger_eps, rng, actor=pol, noise=0.05, beta=beta)
        X, Y = np.vstack([X, Xn]), np.concatenate([Y, Yn])
        pol = train_policy(X, Y, seed=seed, verbose=False)
        r = evaluate(pol, "swing-up", n=10, seed=1000 + i)
        print(f"      round {i + 1}: beta {beta:.3f}, +{len(Xn)} -> {len(X)} samples, swing-up {r['rate'] * 100:.0f}%")
    models["dagger"] = pol

    print("[5] BC version 1: Gaussian policy (NLL, learned sigma), deploy the mean")
    models["bc_nll"] = train_policy(X0, Y0, gaussian=True, seed=seed)

    for k, m in models.items():
        m.save(OUT / f"rotary_bc_{k}")
    print(f"\nsaved -> {OUT}/rotary_bc_*.pt / .npz")
    return models


def load_models():
    out = {k: Policy.load(OUT / f"rotary_bc_{k}") for k in MODELS if (OUT / f"rotary_bc_{k}.npz").exists()}
    if not out:
        raise SystemExit("no saved models -- run:  python3 bc_rotary_swingup.py train")
    return out


# ---------------------------------------------------------------- real-time demo
def demo(model="dagger", gif=None, gif_seconds=24.0):
    import matplotlib
    if gif:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation

    models = load_models()
    order = [k for k in MODELS if k in models] + ["expert"]
    S = {"name": model if model in order else order[0], "paused": False, "last": None}
    rng = np.random.default_rng()

    def build(state=None):
        ctrl = Controller(None if S["name"] == "expert" else models[S["name"]])
        th0, w0, ph0 = state or (math.pi + math.radians(1.0), 0.0, 0.0)
        lp = make_loop(ctrl, th0, w0)
        lp.plant.x[2] = ph0
        S.update(ctrl=ctrl, lp=lp, T=[], TH=[], PH=[], U=[], UE=[], E=[], last=None)

    build()
    fig = plt.figure(figsize=(13.5, 7))
    ax_side = fig.add_axes([0.03, 0.10, 0.36, 0.78])
    ax_top = fig.add_axes([0.28, 0.67, 0.11, 0.21])
    ax_th = fig.add_axes([0.47, 0.70, 0.50, 0.21])
    ax_ph = fig.add_axes([0.47, 0.39, 0.50, 0.21], sharex=ax_th)
    ax_u = fig.add_axes([0.47, 0.08, 0.50, 0.21], sharex=ax_th)

    lim = 1.25 * rm.L
    ax_side.set_aspect("equal"); ax_side.set_xlim(-lim, lim); ax_side.set_ylim(-lim, lim)
    ax_side.set_title("side view (up = balanced)", loc="left")
    ax_side.add_patch(plt.Circle((0, 0), rm.L, fill=False, color="0.85", ls="--"))
    rod, = ax_side.plot([], [], lw=8, solid_capstyle="round")
    ax_side.plot([0], [0], "ko", ms=7)
    title = ax_side.text(0, 1.12 * rm.L, "", ha="center", fontsize=13, weight="bold")
    info = ax_side.text(-0.98 * lim, -0.68 * lim, "", fontsize=8, family="monospace", va="top")

    lim_rad = math.radians(ARM_LIMIT_DEG)
    ax_top.set_aspect("equal"); ax_top.set_xlim(-1.3, 1.3); ax_top.set_ylim(-1.3, 1.3)
    ax_top.set_xticks([]); ax_top.set_yticks([]); ax_top.set_title("arm (top view)", fontsize=8)
    ax_top.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="0.7"))
    for s in (1, -1):
        ax_top.plot([0, s * math.sin(lim_rad)], [0, math.cos(lim_rad)], "r:", lw=0.8)
    arm, = ax_top.plot([], [], lw=4, color="#2774B8")

    line_th, = ax_th.plot([], [], color="#E97820")
    line_ph, = ax_ph.plot([], [], color="#2774B8")
    line_u, = ax_u.plot([], [], color="#C0392B", lw=1.2, label="network command")
    line_ue, = ax_u.plot([], [], color="0.2", lw=0.9, ls="--", label="expert would command")
    ax_th.set(ylabel="theta from up [deg]", ylim=(-190, 190), yticks=[-180, -90, 0, 90, 180])
    ax_th.axhspan(-20, 20, color="#2E8B57", alpha=0.12)
    ax_ph.set(ylabel="arm phi [deg]", ylim=(-160, 160))
    for s in (1, -1):
        ax_ph.axhline(s * ARM_LIMIT_DEG, color="r", ls=":", lw=0.8)
    ax_u.set(ylabel="u [rad/s^2]", xlabel="t [s]", ylim=(-52, 52)); ax_u.legend(fontsize=8, loc="upper right")
    for a in (ax_th, ax_ph, ax_u):
        a.grid(alpha=.3)
    fig.text(0.47, 0.95, "Behavior cloning: 5-128-128-1 ReLU network imitating energy swing-up + pole placement "
             "on the identified rig model (50 Hz policy, 4 ms delay, 0.3 deg encoder, 47 rad/s^2 limit)", fontsize=7.5)

    def on_key(key):
        lp = S["lp"]
        if key == "up": lp.plant.kick(5.0)
        elif key == "left": lp.plant.kick(-1.0)
        elif key == "right": lp.plant.kick(1.0)
        elif key == "c": lp.on = not lp.on
        elif key == " ": S["paused"] = not S["paused"]; S["last"] = None
        elif key == "r": build()
        elif key == "m":
            S["name"] = order[(order.index(S["name"]) + 1) % len(order)]
            x = lp.plant.x.copy()
            build((x[0], x[1], x[2]))
    fig.canvas.mpl_connect("key_press_event", lambda ev: on_key(ev.key))

    script = {8.0: "up", 15.0: "m", 17.0: "up"} if gif else {}
    clock = {"t": 0.0}

    def update(i):
        if gif:
            clock["t"] = i / 20.0
            for kt in [k for k in script if abs(clock["t"] - k) < 1e-6]:
                on_key(script.pop(kt))
        t_now = clock["t"] if gif else time.perf_counter()
        if S["last"] is None:
            S["last"] = t_now
        elapsed = min(t_now - S["last"], 0.1); S["last"] = t_now
        lp, ctrl = S["lp"], S["ctrl"]
        if not S["paused"]:
            lp.advance(int(round(elapsed / rm.DT)))
        th, ph = rm.wrap(lp.plant.x[0]), lp.plant.x[2]
        S["T"].append(lp.plant.t); S["TH"].append(math.degrees(th)); S["PH"].append(math.degrees(ph))
        S["U"].append(lp.u); S["UE"].append(ctrl.a_expert * rm.ACCEL_MAX)
        student = S["name"] != "expert"
        color = "#C0392B" if student else "#2E8B57"
        rod.set_data([0, rm.L * math.sin(th)], [0, rm.L * math.cos(th)]); rod.set_color(color)
        arm.set_data([0, math.sin(ph)], [0, math.cos(ph)])
        title.set_text(f"NETWORK  ({S['name']})" if student else "EXPERT  (energy + pole placement)")
        title.set_color(color)
        info.set_text(f"t={lp.plant.t:6.2f}s  theta={math.degrees(th):7.1f}  arm={math.degrees(ph):7.1f} deg\n"
                      f"u={lp.u:+6.1f} rad/s^2   status={lp.stop or 'OK'}   ctrl={'ON' if lp.on else 'OFF'}"
                      f"{'  PAUSED' if S['paused'] else ''}\n"
                      "keys: up knock over   <- -> kick   m next controller\n"
                      "      c ctrl on/off   r reset   space pause")
        t0 = max(0.0, lp.plant.t - 12.0)
        ax_th.set_xlim(t0, t0 + 12.0)
        line_th.set_data(S["T"], S["TH"]); line_ph.set_data(S["T"], S["PH"])
        line_u.set_data(S["T"], S["U"]); line_ue.set_data(S["T"], S["UE"])
        return rod, arm, line_th, line_ph, line_u, line_ue, info, title

    if gif:
        from matplotlib.animation import PillowWriter
        FuncAnimation(fig, update, frames=int(gif_seconds * 20), blit=False).save(
            gif, writer=PillowWriter(fps=20), dpi=65)
        print(f"gif -> {gif}")
        return
    S["ani"] = FuncAnimation(fig, update, interval=20, cache_frame_data=False)
    plt.show()


def plot_episode(actor, kind="hang", seed=0, ax=None):
    """Static plot of one episode (for Colab): theta, arm and network vs expert command."""
    import matplotlib.pyplot as plt
    r = run_episode(Controller(actor), kind, np.random.default_rng(seed), log=True)
    d = r["log"]
    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(d[:, 0], d[:, 1], color="#E97820"); ax[0].axhspan(-20, 20, color="#2E8B57", alpha=0.12)
    who = "expert" if actor is None else "network"
    result = "balanced" if r["ok"] else "FAILED " + (r["why"] or "")
    up = "" if r["up"] is None else f", up at {r['up']:.2f} s"
    ax[0].set(ylabel="theta from up [deg]", ylim=(-190, 190), yticks=[-180, -90, 0, 90, 180],
              title=f"{who}: {result}{up}")
    ax[1].plot(d[:, 0], d[:, 2], color="#2774B8"); ax[1].set(ylabel="arm [deg]")
    ax[2].plot(d[:, 0], d[:, 3], color="#C0392B", lw=1.2, label="command")
    ax[2].plot(d[:, 0], d[:, 4], "k--", lw=0.8, label="expert would command")
    ax[2].set(ylabel="u [rad/s^2]", xlabel="t [s]"); ax[2].legend(fontsize=8)
    for a in ax:
        a.grid(alpha=.3)
    fig.tight_layout()
    return fig


def animate_episode(actor, kind="hang", seed=0, duration=8.0, fps=20):
    """Animation of one episode for notebooks (Colab has no real-time window).
    Returns a matplotlib FuncAnimation; show it with IPython.display.HTML(anim.to_jshtml())."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    r = run_episode(Controller(actor), kind, np.random.default_rng(seed), duration=duration, log=True)
    d = r["log"]                                       # t, theta deg, arm deg, u, expert u  (every 10 ms)
    step = max(1, int(round(1.0 / (fps * 0.01))))
    frames = range(0, len(d), step)
    fig, (ax, ax_u) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw=dict(width_ratios=[1, 1.6]))
    lim = 1.25 * rm.L
    ax.set_aspect("equal"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.axis("off")
    ax.add_patch(plt.Circle((0, 0), rm.L, fill=False, color="0.85", ls="--"))
    rod, = ax.plot([], [], lw=7, color="#C0392B" if actor is not None else "#2E8B57", solid_capstyle="round")
    ax.plot([0], [0], "ko", ms=6)
    txt = ax.text(0, -1.15 * rm.L, "", ha="center", fontsize=9, family="monospace")
    ax_u.plot(d[:, 0], d[:, 1], color="#E97820", lw=1.2)
    ax_u.axhspan(-20, 20, color="#2E8B57", alpha=0.12)
    ax_u.set(xlabel="t [s]", ylabel="theta from up [deg]", ylim=(-190, 190), yticks=[-180, -90, 0, 90, 180])
    ax_u.grid(alpha=.3)
    cursor = ax_u.axvline(0, color="k", lw=0.8)
    ax.set_title(f"{'expert' if actor is None else 'network'} — {'balanced' if r['ok'] else 'failed'}")

    def update(i):
        th = math.radians(d[i, 1])
        rod.set_data([0, rm.L * math.sin(th)], [0, rm.L * math.cos(th)])
        txt.set_text(f"t={d[i, 0]:4.1f}s  theta={d[i, 1]:6.1f}  arm={d[i, 2]:6.1f}")
        cursor.set_xdata([d[i, 0], d[i, 0]])
        return rod, txt, cursor

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)
    plt.close(fig)
    return anim


def compare_animation(actors, kind="top", seed=3, duration=8.0, fps=20):
    """Side-by-side animation of several controllers in the SAME episode (same start, same kick).
    actors: {"name": policy or None (= expert)}.  Returns a FuncAnimation."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation
    logs = {}
    for name, act in actors.items():
        r = run_episode(Controller(act), kind, np.random.default_rng(seed), duration=duration, log=True)
        logs[name] = (r, r["log"])
    n = len(actors)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 3.7))
    lim = 1.25 * rm.L
    rods, texts = [], []
    for ax, (name, (r, d)) in zip(np.atleast_1d(axes), logs.items()):
        ax.set_aspect("equal"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.axis("off")
        ax.add_patch(plt.Circle((0, 0), rm.L, fill=False, color="0.85", ls="--"))
        rod, = ax.plot([], [], lw=6, solid_capstyle="round",
                       color="#2E8B57" if actors[name] is None else "#C0392B")
        ax.plot([0], [0], "ko", ms=5)
        ax.set_title(f"{name}\n{'balanced' if r['ok'] else 'FAILED'}", fontsize=10,
                     color="k" if r["ok"] else "#C0392B")
        rods.append((rod, d)); texts.append(ax.text(0, -1.18 * rm.L, "", ha="center", fontsize=8))
    step = max(1, int(round(1.0 / (fps * 0.01))))
    n_frames = min(len(d) for _, d in rods)

    def update(i):
        k = min(i * step, n_frames - 1)
        for (rod, d), txt in zip(rods, texts):
            th = math.radians(d[k, 1])
            rod.set_data([0, rm.L * math.sin(th)], [0, rm.L * math.cos(th)])
            txt.set_text(f"t={d[k, 0]:4.1f}s  arm {d[k, 2]:5.0f}°")
        return [r for r, _ in rods] + texts

    fig.suptitle({"top": "knocked over at 2 s (4-6 rad/s): does it swing up again?",
                  "hang": "swing-up from hanging"}[kind], fontsize=10, y=0.98)
    fig.tight_layout(rect=(0, 0, 1, 0.9))
    anim = FuncAnimation(fig, update, frames=n_frames // step, interval=1000 / fps, blit=True)
    plt.close(fig)
    return anim


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("cmd", choices=("train", "eval", "demo", "gif"))
    ap.add_argument("--model", default="dagger", choices=MODELS + ("expert",))
    ap.add_argument("--demos", type=int, default=20)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    if args.cmd == "train":
        compare(train_all(n_demo=args.demos, seed=args.seed))
    elif args.cmd == "eval":
        compare(load_models())
    elif args.cmd == "demo":
        demo(args.model)
    else:
        demo(args.model, gif=str(OUT / "swingup_demo.gif"))


if __name__ == "__main__":
    main()


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from IPython.display import HTML
import importlib, bc_rotary_swingup as B
importlib.reload(B)
rng = np.random.default_rng(0)
print("선생님 게인 K =", np.round(B.K, 2))

## 1. 선생님(전문가) 제어기
상태만 보고 정답을 주는 제어기입니다 (DAgger는 아무 상태에서나 정답이 필요해서 '기억 없는' 버전).
- 꼭대기 근처 (|θ| < 20°, |θ̇| < 5 rad/s): 4상태 피드백 `u = −K x`
- 그 밖: 에너지 펌핑 `u = 30·clip(0.3(1−E),0,1)·sign(θ̇ cosθ) − 5φ̇ − 3φ`

In [ ]:
B.plot_episode(None, "hang"); plt.show()      # 매달림 → 흔들어 올려 세우기
HTML(B.animate_episode(None, "hang").to_jshtml())

## 2. 시연 데이터 수집
선생님이 20번 시연합니다 (절반은 매달림에서 올리기, 절반은 세운 상태에서 쓰러뜨린 뒤 다시 올리기). 정책 한 스텝(20 ms)마다 `(관측, 선생님 행동)`을 저장합니다.
- 관측 `obs = [cos θ, sin θ, θ̇, φ, φ̇]`
- 행동 `a = u / 47.1` ∈ [−1, 1]

In [ ]:
X0, Y0 = B.collect(20, rng)
print("데이터:", X0.shape, Y0.shape)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(np.degrees(np.arctan2(X0[:, 1], X0[:, 0])), X0[:, 2], s=1, c=Y0, cmap="coolwarm")
ax[0].set(xlabel="theta from up [deg]", ylabel="theta' [rad/s]", title="states visited by the expert (colour = expert action)")
ax[1].hist(Y0, bins=50); ax[1].set(xlabel="expert action a", title="expert action distribution")
plt.tight_layout(); plt.show()

## 3. BC 학습 (version 0: MSE 회귀)
신경망 5→128→128→1 (ReLU), 출력 `tanh`. 선생님 행동과의 **평균제곱오차**를 최소화합니다. (Lec05: 출력에 가우시안 잡음을 가정한 최대우도 = MSE)

In [ ]:
bc = B.train_policy(X0, Y0)
B.plot_episode(bc, "hang"); plt.show()

## 4. compounding error 확인
시연과 비슷한 상황(스윙업)은 되지만, **쓰러뜨리기**처럼 시연에서 드물었던 상황이나 **지연**이 커지면 작은 실수가 쌓여 실패하기 쉽습니다.

In [ ]:
for test in B.TESTS:
    r = B.evaluate(bc, test, n=10)
    print(f"BC  {test:<12} 성공 {r['rate']*100:3.0f}%")
B.plot_episode(bc, "top", seed=3); plt.show()

## 5. DART — 시연에 노이즈 섞기
선생님이 행동할 때 일부러 노이즈(σ=0.2)를 섞어 **조금 벗어난 상태**도 방문하게 하고, 거기에 깨끗한 선생님 정답을 붙입니다.

In [ ]:
Xd, Yd = B.collect(20, rng, noise=0.2)
bc_dart = B.train_policy(np.vstack([X0, Xd]), np.concatenate([Y0, Yd]))

## 6. DAgger — 학생이 간 상태에 선생님 정답 붙이기
1. 학생(현재 신경망)이 직접 조종 (확률 β로 선생님이 대신 조종, 라운드마다 β 절반)
2. 학생이 방문한 상태마다 선생님 정답을 기록
3. 데이터에 합치고 다시 학습 → 반복

In [ ]:
X, Y, pol = X0.copy(), Y0.copy(), bc
for i in range(5):
    beta = 0.5 ** (i + 1)
    Xn, Yn = B.collect(6, rng, actor=pol, noise=0.05, beta=beta)   # 학생이 조종, 선생님이 정답
    X, Y = np.vstack([X, Xn]), np.concatenate([Y, Yn])             # 데이터 합치기
    pol = B.train_policy(X, Y, verbose=False)                     # 다시 학습
    r = B.evaluate(pol, "knock-over", n=10, seed=100 + i)
    print(f"round {i+1}: beta {beta:.3f}, 데이터 {len(X)}개, 쓰러뜨리기 회복 {r['rate']*100:.0f}%")
dagger = pol

## 7. 비교
(version 1: 평균과 σ를 함께 출력해 로그확률을 최대화하는 가우시안 정책도 함께 비교합니다.)

In [ ]:
bc_nll = B.train_policy(X0, Y0, gaussian=True)
B.compare({"bc": bc, "bc_dart": bc_dart, "dagger": dagger, "bc_nll": bc_nll}, n=10)

### 같은 상황에서 다섯 가지 비교 (영상)
선생님 / BC / BC+DART / DAgger / BC(가우시안 NLL)이 **똑같은 시작과 똑같은 충격**을 받습니다.

In [ ]:
actors = {"expert": None, "bc": bc, "bc_dart": bc_dart, "dagger": dagger, "bc_nll": bc_nll}
HTML(B.compare_animation(actors, "top", seed=5).to_jshtml())    # 2초에 쓰러뜨림 → 다시 올리나?

In [ ]:
HTML(B.compare_animation(actors, "hang", seed=0).to_jshtml())   # 매달림에서 스윙업

## 8. weight 저장 → 실제 장비
저장 형식은 `swing-up/swingup_rl_control.py --kind bc` 와 같습니다 (입력 `[cos θ, sin θ, θ̇, φ, φ̇]`, ReLU MLP, `net.N.*`, 출력 `tanh × 47.1 rad/s²`).

In [ ]:
dagger.save("rotary_bc_dagger"); bc_dart.save("rotary_bc_bc_dart"); bc.save("rotary_bc_bc")
try:
    from google.colab import files
    files.download("rotary_bc_dagger.pt")
except ImportError:
    print("Colab이 아니어서 다운로드는 건너뜀 — rotary_bc_*.pt 가 현재 폴더에 저장됨")

실제 장비 (진자를 매단 채 보드 리셋 후):
```bash
python3 swing-up/swingup_rl_control.py --sim --yes --model rotary_bc_dagger.pt --kind bc   # 먼저 시뮬로 확인
python3 swing-up/swingup_rl_control.py --model rotary_bc_dagger.pt --kind bc               # 실제
```